# Figure 6C — APC activation and maturation markers in MHC II high vs low LUAD

**Goal:** Compare expression of APC activation and maturation marker genes
across myeloid cell types between MHC II high and low LUAD tumors. Two
resolutions are used: `cell_type_major` for the primary figure (consistent
with other analyses) and `ann_fine` for the classical/non-classical monocyte
split. APC UMAP is computed to identify subpopulations not captured by
bulk marker analysis.

**Input:** `outputs/processed/luad_mhc2_classified.h5ad`
**Output:** Figure 6C saved to `outputs/figures/figure6/`

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests
from pathlib import Path
import yaml
from ceiba.plot_utils import plot_dual_metric_panel

sns.set(font_scale=2)
sns.set_style('ticks')
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

palette    = {'MHC class II High': '#FF8811FF', 'MHC class II Low': '#462255FF'}
group_order = ['MHC class II High', 'MHC class II Low']

In [ ]:
with open('/home/gh8sj/projects/mhc2-luad-paper/data/paths_config.yml') as f:
    cfg = yaml.safe_load(f)

repo_root = Path(cfg['repo_root'])
fig_out   = repo_root / cfg['outputs']['figures'] / 'figure6'
table_out = repo_root / cfg['outputs']['tables']  / 'figure6'

fig_out.mkdir(parents=True, exist_ok=True)
table_out.mkdir(parents=True, exist_ok=True)

In [ ]:
tumordata = sc.read_h5ad(repo_root / 'outputs/processed/luad_mhc2_classified.h5ad')

tumordata = tumordata[
    tumordata.obs['MHC2_clustering'].isin(['MHC class II High', 'MHC class II Low'])
].copy()

print(f"Cells: {tumordata.n_obs:,}")
print(tumordata.obs['MHC2_clustering'].value_counts())
print(tumordata.obs['cell_type_major'].value_counts())

## APC marker gene selection

Marker genes selected to characterize APC activation state across
three functional categories:

- **DC maturation/migration:** LAMP3 (DC-LAMP), CCR7 (lymph node homing),
  FSCN1 (migratory DC actin bundling)
- **Co-stimulatory molecules:** CD80, CD86, CD40 — required for productive
  T cell activation
- **Chemokines produced by activated APCs:** CCL17, CCL22 (CCR4 ligands)
- **Monocyte markers:** CD14, FCGR3A (classical vs non-classical),
  S100A8/S100A9 (inflammatory), CCL2, CX3CR1 (recruitment/residency)
- **Inflammatory mediators:** IL1B, TNF

The hypothesis being tested: MHC II high tumors have activated APCs
(DC maturation markers enriched), while MHC II low tumors may have
a more inflammatory monocyte state (S100A8/9, IL1B enriched).

In [ ]:
apc_genes_of_interest = [
    # DC maturation / APC activation
    'LAMP3', 'CCR7', 'FSCN1', 'CD80', 'CD86', 'CD40',
    # chemokines produced by activated APCs
    'CCL17', 'CCL22',
    # monocyte markers
    'CD14', 'FCGR3A',
    'S100A8', 'S100A9',
    'IL1B', 'TNF',
    'CCL2', 'CX3CR1',
]

apc_genes_dict = {}
for g in apc_genes_of_interest:
    match = tumordata.var[tumordata.var['feature_name'] == g]
    if len(match) > 0:
        apc_genes_dict[g] = list(match.index)
    else:
        print(f'Warning: {g} not found')

print(f"Genes mapped: {list(apc_genes_dict.keys())}")

## Figure 6C — Primary figure (cell_type_major resolution)

Primary figure using `cell_type_major` for consistency with other
analyses in the paper. Cell types: Macrophage, Monocyte, cDC1, cDC2,
DC mature. Neutrophils excluded — no significant findings in either
marker expression or cell type abundance.

Dual metric: percent expressing (top) and mean expression among
positive cells (bottom). FDR correction across all gene × cell type
combinations using Benjamini-Hochberg.

In [ ]:
apc_cell_types_primary = ['Macrophage', 'Monocyte', 'cDC1', 'cDC2', 'DC mature']

stats_apc = plot_dual_metric_panel(
    tumordata,
    genes_dict=apc_genes_dict,
    cell_types=apc_cell_types_primary,
    group_col='MHC2_clustering',
    group_order=group_order,
    palette=palette,
    fig_path=fig_out / 'fig6c_apc_markers_primary.pdf',
    title='APC activation and maturation markers by MHC II classification',
)
stats_apc.to_csv(table_out / 'fig6c_apc_marker_stats.csv', index=False)
print(stats_apc[['cell_type', 'gene', 'FDR_p', 'sig_label']].to_string(index=False))

In [ ]:
neutrophil_genes_dict

## Supplementary — Neutrophil marker analysis (negative result)

Neutrophil and monocyte inflammatory marker genes were tested to
investigate whether MHC II low tumors show a distinct inflammatory
innate immune state. No significant differences survived FDR correction
in either neutrophils or monocytes for these markers. This negative
result is documented here for transparency and suggests the divergent
innate immune response hypothesis is not detectable at the
transcriptional level in scRNA-seq data, possibly due to low neutrophil
capture or protein-level regulation.

In [ ]:
neutrophil_genes_found = [
    'CXCR1', 'CXCR2', 'FCGR3B', 'FCGR3A',
    'MPO', 'ELANE', 'AZU1', 'CTSG',
    'IL1B', 'IL6', 'TNF', 'CXCL8',
    'S100A8', 'S100A9', 'S100A12',
    'CCR1', 'CCR2', 'CXCR4',
    'MMP8', 'MMP9', 'LCN2', 'CAMP',
]

neutrophil_genes_dict = {}
for g in neutrophil_genes_found:
    match = tumordata.var[tumordata.var['feature_name'] == g]
    if len(match) > 0:
        neutrophil_genes_dict[g] = list(match.index)
    else:
        print(f'Warning: {g} not found')

print(f"Neutrophil genes mapped: {list(neutrophil_genes_dict.keys())}")

# first rerun neutrophil stats with dual metrics
neutrophil_stats = []
neutrophil_pct  = {}
neutrophil_mean = {}

for cell_type in ['Neutrophils', 'Monocyte']:
    subset = tumordata[tumordata.obs['cell_type_major'] == cell_type]
    for gene, gene_ens_list in neutrophil_genes_dict.items():
        gene_ens = gene_ens_list[0]
        x = subset.to_df()[gene_ens]
        expr = x.to_frame(name='expr')
        expr['sample']  = subset.obs['sample'].values
        expr['cluster'] = subset.obs['MHC2_clustering'].values

        pct_df = (
            expr.assign(detected=(expr['expr'] > 0).astype(int))
            .groupby(['sample', 'cluster'], observed=True)['detected']
            .mean().mul(100).reset_index()
            .rename(columns={'detected': 'expr'})
        )
        mean_df = (
            expr[expr['expr'] > 0]
            .groupby(['sample', 'cluster'], observed=True)['expr']
            .mean().reset_index()
        )
        neutrophil_pct[(cell_type, gene)]  = pct_df
        neutrophil_mean[(cell_type, gene)] = mean_df

        g1_pct  = pct_df.loc[pct_df['cluster'] == group_order[0], 'expr']
        g2_pct  = pct_df.loc[pct_df['cluster'] == group_order[1], 'expr']
        g1_mean = mean_df.loc[mean_df['cluster'] == group_order[0], 'expr']
        g2_mean = mean_df.loc[mean_df['cluster'] == group_order[1], 'expr']

        p_pct  = mannwhitneyu(g1_pct,  g2_pct,  alternative='two-sided')[1] if len(g1_pct)  > 0 and len(g2_pct)  > 0 else np.nan
        p_mean = mannwhitneyu(g1_mean, g2_mean, alternative='two-sided')[1] if len(g1_mean) > 0 and len(g2_mean) > 0 else np.nan

        neutrophil_stats.append({
            'cell_type': cell_type, 'gene': gene,
            'p_pct': p_pct, 'p_mean': p_mean,
            'mean_high_pct':  g1_pct.mean(),  'mean_low_pct':  g2_pct.mean(),
            'mean_high_expr': g1_mean.mean(), 'mean_low_expr': g2_mean.mean(),
        })

neutrophil_stats_df = pd.DataFrame(neutrophil_stats)

# FDR within each cell type
fdr_results = []
for ct, sub in neutrophil_stats_df.groupby('cell_type', observed=True):
    sub = sub.copy()
    _, fdr_pct,  _, _ = multipletests(sub['p_pct'].fillna(1),  method='fdr_bh')
    _, fdr_mean, _, _ = multipletests(sub['p_mean'].fillna(1), method='fdr_bh')
    sub['FDR_pct']  = fdr_pct
    sub['FDR_mean'] = fdr_mean
    fdr_results.append(sub)

neutrophil_stats_df = pd.concat(fdr_results)
neutrophil_stats_df['sig_pct']  = neutrophil_stats_df['FDR_pct'].apply(
    lambda p: '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else '')
neutrophil_stats_df['sig_mean'] = neutrophil_stats_df['FDR_mean'].apply(
    lambda p: '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else '')
neutrophil_stats_df['direction'] = neutrophil_stats_df.apply(
    lambda r: 'MHC II High' if r['mean_high_pct'] > r['mean_low_pct'] else 'MHC II Low', axis=1)
neutrophil_stats_df['-log10_FDR_pct']  = -np.log10(neutrophil_stats_df['FDR_pct'].clip(lower=1e-10))
neutrophil_stats_df['-log10_FDR_mean'] = -np.log10(neutrophil_stats_df['FDR_mean'].clip(lower=1e-10))

neutrophil_stats_df.to_csv(table_out / 'figS_neutrophil_monocyte_stats.csv', index=False)
print(neutrophil_stats_df[['cell_type', 'gene', 'sig_pct', 'sig_mean', 'direction']].to_string(index=False))

# plot — 2 rows (cell types) × 2 cols (metrics)
fig, axes = plt.subplots(2, 2, figsize=(14, len(neutrophil_genes_found) * 0.35 * 2 + 2),
                          sharey=False)

for r, cell_type in enumerate(['Neutrophils', 'Monocyte']):
    sub = neutrophil_stats_df[neutrophil_stats_df['cell_type'] == cell_type]

    for c, (metric, col, sig_col) in enumerate([
        ('% cells expressing',          '-log10_FDR_pct',  'sig_pct'),
        ('Mean expression (pos cells)', '-log10_FDR_mean', 'sig_mean'),
    ]):
        ax = axes[r, c]
        plot_df = sub.sort_values(col, ascending=True)

        for _, row in plot_df.iterrows():
            color  = palette['MHC class II High'] if row['direction'] == 'MHC II High' else palette['MHC class II Low']
            sig    = row[sig_col] != ''
            marker = 'o' if sig else 'x'
            ax.plot([0, row[col]], [row['gene'], row['gene']],
                    color=color, lw=1.5, alpha=0.6)
            ax.scatter(row[col], row['gene'],
                       color=color, s=80 if sig else 50,
                       marker=marker, zorder=3, linewidths=1.2)

        ax.axvline(-np.log10(0.05), color='gray', linestyle='--', lw=1)
        ax.set_xlabel('-log10(FDR-adjusted p-value)', fontsize=11)
        if c == 0:
            ax.set_ylabel(cell_type, fontsize=12)
        ax.set_title(metric if r == 0 else '', fontsize=12)
        ax.spines[['top', 'right']].set_visible(False)

legend_elements = [
    Line2D([0], [0], marker='o', color='w',
           markerfacecolor=palette['MHC class II High'],
           markersize=10, label='Higher in MHC II High'),
    Line2D([0], [0], marker='o', color='w',
           markerfacecolor=palette['MHC class II Low'],
           markersize=10, label='Higher in MHC II Low'),
    Line2D([0], [0], marker='o', color='gray',
           markerfacecolor='gray', markersize=10, label='FDR < 0.05'),
    Line2D([0], [0], marker='x', color='gray',
           markersize=8, label='ns', linewidth=0),
]
fig.legend(handles=legend_elements, loc='upper center', ncol=4,
           frameon=False, bbox_to_anchor=(0.5, 1.02), fontsize=10)
fig.suptitle('Neutrophil and monocyte inflammatory markers (supplementary)',
             fontsize=13, y=1.05)
plt.tight_layout()
plt.savefig(fig_out / 'figS_neutrophil_monocyte_lollipop.pdf', bbox_inches='tight')
plt.show()

## APC UMAP — subpopulation analysis

Subset to APC cell types and compute new UMAP coordinates with Harmony
batch correction by study. Leiden clustering identifies subpopulations
not visible in bulk marker analysis. Density plots compare MHC II high
vs low cell distributions across the UMAP.

Key questions:
- Are there APC subpopulations specifically enriched in MHC II high or low?
- Does the classical/non-classical monocyte split show spatial separation?
- Where do DC mature cells sit relative to cDC1/cDC2?

In [ ]:
%%time

import harmonypy

apc_broad = ['Macrophage', 'Monocyte', 'cDC1', 'cDC2', 'DC mature', 'pDC']
apc = tumordata[tumordata.obs['cell_type_major'].isin(apc_broad)].copy()
print(f"APC cells: {apc.n_obs:,}")
print(apc.obs['cell_type_major'].value_counts())

apc_path = repo_root / 'outputs/processed/apc_mhc2_umap.h5ad'

if apc_path.exists():
    apc = sc.read_h5ad(apc_path)
    print("Loaded from checkpoint")
else:
    sc.pp.highly_variable_genes(apc, n_top_genes=2000, batch_key='study')
    sc.pp.pca(apc, n_comps=50, use_highly_variable=True)
    ho = harmonypy.run_harmony(apc.obsm['X_pca'].astype(np.float64),
                           apc.obs, 'study')
    apc.obsm['X_pca_harmony'] = ho.Z_corr.T if ho.Z_corr.shape[1] == apc.n_obs else ho.Z_corr
    sc.pp.neighbors(apc, use_rep='X_pca_harmony', n_neighbors=30)
    sc.tl.umap(apc)
    sc.tl.leiden(apc, resolution=0.5)
    apc.write_h5ad(apc_path)
    print("Computed and saved checkpoint")

In [ ]:
sc.settings.figdir = str(fig_out)

# cell type
sc.pl.umap(apc, color='cell_type_major', title='APC cell types',
           frameon=False, legend_loc='on data',
           save='_fig6c_apc_cell_types.pdf')

# ann_fine for monocyte split
sc.pl.umap(apc, color='ann_fine', title='APC subtypes (ann_fine)',
           frameon=False, legend_loc='on data',
           save='_fig6c_apc_ann_fine.pdf')

# MHC2 classification
sc.pl.umap(apc, color='MHC2_clustering',
           palette=palette,
           title='APC — MHC II classification',
           frameon=False,
           save='_fig6c_apc_mhc2_clustering.pdf')

# leiden clusters
sc.pl.umap(apc, color='leiden', legend_loc='on data',
           title='APC — Leiden clusters',
           frameon=False,
           save='_fig6c_apc_leiden.pdf')

In [ ]:
sc.tl.embedding_density(apc, groupby='MHC2_clustering')
sc.pl.embedding_density(
    apc, groupby='MHC2_clustering', ncols=2,
    save='_fig6c_apc_density.pdf'
)

In [ ]:
apc_log = apc.copy()
sc.pp.normalize_total(apc_log, target_sum=1e4)
sc.pp.log1p(apc_log)
sc.tl.rank_genes_groups(apc_log, groupby='leiden', method='wilcoxon')
sc.pl.rank_genes_groups(apc_log, n_genes=15, sharey=False,
                        gene_symbols='feature_name',
                        save='_fig6c_apc_leiden_markers.pdf')

In [ ]:
rank_df = sc.get.rank_genes_groups_df(apc_log, group=None)
rank_df

In [ ]:
cluster_props = (
    apc.obs.groupby(['MHC2_clustering', 'leiden'], observed=True)
    .size()
    .groupby(level=0)
    .transform(lambda x: x / x.sum() * 100)
    .reset_index(name='pct')
)
print(
    cluster_props.pivot(index='leiden', columns='MHC2_clustering', values='pct')
    .round(1)
    .to_string()
)

In [ ]:
cluster_labels = {
    '0': 'Tissue-resident mac\n(APOE+C1Q+)',
    '1': 'SPP1+ mac',
    '2': 'Activated DC\n(HLA-DR+CLEC10A+)',
    '3': 'Inflammatory mono\n(FCN1+S100A8+)',
    '4': 'Alveolar mac\n(MRC1+MARCO+)',
    '5': 'Plasma/doublets',
    '6': 'Low quality',
    '7': 'CD1c+ DC',
    '8': 'Tolerogenic DC\n(IDO1+)',
    '9': 'pDC (LILRA4+IRF7+)',
}

prop_df = (
    apc.obs.groupby(['MHC2_clustering', 'leiden'], observed=True)
    .size()
    .groupby(level=0)
    .transform(lambda x: x / x.sum() * 100)
    .reset_index(name='pct')
)
prop_df['cluster_label'] = prop_df['leiden'].map(cluster_labels)

pivot = prop_df.pivot(index='MHC2_clustering', columns='cluster_label', values='pct')

fig, ax = plt.subplots(figsize=(9, 5))
pivot.plot(kind='bar', stacked=True, ax=ax,
           colormap='tab10', edgecolor='white', linewidth=0.5)
ax.set_xlabel('')
ax.set_ylabel('% of APC cells')
ax.set_xticklabels(['MHC II High', 'MHC II Low'], rotation=0)
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', frameon=False, fontsize=9)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig(fig_out / 'fig6c_apc_cluster_proportions.pdf', bbox_inches='tight')
plt.show()

In [ ]:
for cluster, label in [('2', 'Activated DC (HLA-DR+CLEC10A+)'),
                        ('3', 'Inflammatory monocyte (FCN1+S100A8+)'),
                        ('9', 'pDC (LILRA4+IRF7+)')]:
    cell_cluster = (
        apc.obs.groupby(['sample', 'MHC2_clustering', 'leiden'], observed=True)
        .size().reset_index(name='count')
    )
    total = apc.obs.groupby(['sample', 'MHC2_clustering'], observed=True).size().reset_index(name='total')
    merged = cell_cluster.merge(total, on=['sample', 'MHC2_clustering'])
    merged['pct'] = merged['count'] / merged['total'] * 100
    sub = merged[
        (merged['leiden'] == cluster) &
        (merged['MHC2_clustering'].isin(group_order))
    ]
    g1 = sub.loc[sub['MHC2_clustering'] == 'MHC class II High', 'pct']
    g2 = sub.loc[sub['MHC2_clustering'] == 'MHC class II Low',  'pct']
    _, p = mannwhitneyu(g1, g2, alternative='two-sided')
    sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
    print(f'Cluster {cluster} ({label}): p={p:.3e} ({sig})  mean_high={g1.mean():.1f}%  mean_low={g2.mean():.1f}%')

In [ ]:
# boxplot for cluster 9 pDC
for cluster, label, fname in [
    ('9', 'pDC (LILRA4+IRF7+)', 'fig6c_pdc_enrichment.pdf'),
]:
    cell_cluster = (
        apc.obs.groupby(['sample', 'MHC2_clustering', 'leiden'], observed=True)
        .size().reset_index(name='count')
    )
    total = apc.obs.groupby(['sample', 'MHC2_clustering'], observed=True).size().reset_index(name='total')
    merged = cell_cluster.merge(total, on=['sample', 'MHC2_clustering'])
    merged['pct'] = merged['count'] / merged['total'] * 100
    sub = merged[
        (merged['leiden'] == cluster) &
        (merged['MHC2_clustering'].isin(group_order))
    ].copy()

    g1 = sub.loc[sub['MHC2_clustering'] == 'MHC class II High', 'pct']
    g2 = sub.loc[sub['MHC2_clustering'] == 'MHC class II Low',  'pct']
    _, p = mannwhitneyu(g1, g2, alternative='two-sided')
    sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'

    fig, ax = plt.subplots(figsize=(4, 5))
    sns.boxplot(
        data=sub, x='MHC2_clustering', y='pct', hue='MHC2_clustering',
        order=group_order, palette=palette,
        ax=ax, fill=False, showfliers=False, legend=False,
    )
    sns.stripplot(
        data=sub, x='MHC2_clustering', y='pct', hue='MHC2_clustering',
        order=group_order, palette=palette,
        ax=ax, edgecolor='k', linewidth=1, size=5, alpha=0.7, legend=False,
    )
    ax.text(0.5, 0.88, f'{sig}\np={p:.3e}', ha='center', va='bottom',
            transform=ax.transAxes, fontsize=12)
    ax.set_title(f'pDC enrichment\n(LILRA4+IRF7+)', fontsize=12)
    ax.set_ylabel('% of APC cells')
    ax.set_xlabel('')
    ax.set_xticklabels([])
    ax.spines[['top', 'right']].set_visible(False)
    plt.tight_layout()
    plt.savefig(fig_out / fname, bbox_inches='tight')
    plt.show()